# Land Cover Classification using Machine Learning

This notebook demonstrates satellite image classification for land cover analysis using Random Forest and KNN classifiers.

**Dataset**: DeepGlobe Land Cover Classification

**Classes**: Urban, Agriculture, Rangeland, Forest, Water, Barren, Unknown

In [ ]:
# ==========================================
# IMPORTS & SETUP
# ==========================================
import numpy as np
import pandas as pd
import cv2
import os
import glob
import warnings
import time
import pickle
from joblib import Parallel, delayed

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)

print("✓ Libraries imported successfully")

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
class Config:
    """Configuration parameters for the classification pipeline"""
    
    # Path Configuration
    BASE_SEARCH_PATH = "/kaggle/input"
    OUTPUT_PATH = "/kaggle/working/outputs"
    
    # Image preprocessing
    IMG_SIZE = (512, 512)
    
    # DeepGlobe class definitions (RGB to class index mapping)
    COLOR_DICT = {
        (0, 255, 255): 0,      # Urban
        (255, 255, 0): 1,      # Agriculture
        (255, 0, 255): 2,      # Rangeland
        (0, 255, 0): 3,        # Forest
        (0, 0, 255): 4,        # Water
        (255, 255, 255): 5,    # Barren
        (0, 0, 0): 6           # Unknown
    }
    
    CLASS_NAMES = ['Urban', 'Agriculture', 'Rangeland', 'Forest', 'Water', 'Barren', 'Unknown']
    CLASS_COLORS = np.array([
        [0, 255, 255], [255, 255, 0], [255, 0, 255], [0, 255, 0],
        [0, 0, 255], [255, 255, 255], [0, 0, 0]
    ]) / 255.0
    
    # Data sampling parameters
    SAMPLE_RATE = 0.10      # Sample 10% of pixels per image
    MAX_IMAGES = 100        # Maximum number of images to process
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    
    # Feature extraction flags
    USE_TEXTURE_FEATURES = True
    USE_SPECTRAL_INDICES = True
    TEXTURE_WINDOW = 3

os.makedirs(Config.OUTPUT_PATH, exist_ok=True)
print(f"✓ Configuration loaded successfully")
print(f"  - Target Size: {Config.IMG_SIZE}")
print(f"  - Sample Rate: {Config.SAMPLE_RATE}")

In [ ]:
# ==========================================
# DATA DISCOVERY
# ==========================================
def discover_dataset(base_path):
    """
    Discover and validate satellite image and mask pairs in the dataset.
    
    Args:
        base_path (str): Root directory to search for images
        
    Returns:
        tuple: Lists of satellite image paths and corresponding mask paths
    """
    print(f"Searching for data in {base_path}...")
    sat_files = sorted(glob.glob(os.path.join(base_path, "**", "*_sat.jpg"), recursive=True))
    
    # Fallback search if standard naming not found
    if not sat_files:
        sat_files = sorted(glob.glob(os.path.join(base_path, "**", "*.jpg"), recursive=True))
        sat_files = [f for f in sat_files if "mask" not in f.lower()]

    if not sat_files:
        raise FileNotFoundError("❌ No images found! Please verify the dataset path.")

    mask_files = []
    valid_sat = []
    
    # Validate that each satellite image has a corresponding mask
    for s in sat_files:
        candidates = [s.replace('_sat.jpg', '_mask.png'), s.replace('.jpg', '_mask.png')]
        for m in candidates:
            if os.path.exists(m):
                valid_sat.append(s)
                mask_files.append(m)
                break
                
    print(f"✓ Found {len(valid_sat)} valid image-mask pairs")
    return valid_sat, mask_files

sat_files, mask_files = discover_dataset(Config.BASE_SEARCH_PATH)

In [ ]:
# ==========================================
# FEATURE ENGINEERING
# ==========================================
def compute_spectral_indices(img):
    """
    Calculate spectral indices from RGB bands.
    
    Args:
        img (numpy.ndarray): RGB image
        
    Returns:
        numpy.ndarray: Stack of spectral indices (NDVI, brightness)
    """
    eps = 1e-8
    r = img[:,:,0].astype(np.float32) + eps
    g = img[:,:,1].astype(np.float32) + eps
    b = img[:,:,2].astype(np.float32) + eps
    
    # Normalized Difference Vegetation Index (proxy using visible bands)
    ndvi = (g - r) / (g + r)
    
    # Overall brightness
    brightness = (r + g + b) / 3.0
    
    return np.dstack([ndvi, brightness])

def compute_texture_fast(img, window=3):
    """
    Compute texture features using local standard deviation and range.
    
    Args:
        img (numpy.ndarray): Input RGB image
        window (int): Window size for texture calculation
        
    Returns:
        numpy.ndarray: Stack of texture features (std dev, range)
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    
    # Local standard deviation
    mean = cv2.blur(gray, (window, window))
    mean_sq = cv2.blur(gray**2, (window, window))
    std = np.sqrt(np.maximum(mean_sq - mean**2, 0))
    
    # Local range (max - min)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (window, window))
    rng = cv2.dilate(gray, kernel) - cv2.erode(gray, kernel)
    
    return np.dstack([std, rng])

def extract_features(img):
    """
    Extract all configured features from an image.
    
    Args:
        img (numpy.ndarray): Input RGB image
        
    Returns:
        numpy.ndarray: Concatenated feature stack
    """
    feats = [img.astype(np.float32)]
    if Config.USE_SPECTRAL_INDICES:
        feats.append(compute_spectral_indices(img))
    if Config.USE_TEXTURE_FEATURES:
        feats.append(compute_texture_fast(img, Config.TEXTURE_WINDOW))
    return np.dstack(feats)

def rgb_mask_to_label(mask, color_dict):
    """
    Convert RGB mask to class labels.
    
    Args:
        mask (numpy.ndarray): RGB mask image
        color_dict (dict): Mapping from RGB tuples to class indices
        
    Returns:
        numpy.ndarray: 2D array of class labels
    """
    h, w = mask.shape[:2]
    label = np.full((h, w), 6, dtype=np.uint8)  # Default to 'Unknown'
    for col, idx in color_dict.items():
        match = np.all(mask == col, axis=-1)
        label[match] = idx
    return label

In [ ]:
# ==========================================
# DATA PREPARATION (PARALLEL PROCESSING)
# ==========================================
def process_one(idx, sat_p, mask_p):
    """
    Process a single image-mask pair: resize, extract features, and sample pixels.
    
    Args:
        idx (int): Image index
        sat_p (str): Path to satellite image
        mask_p (str): Path to mask image
        
    Returns:
        tuple: (features, labels) or None if processing fails
    """
    try:
        # Load and resize images
        sat = cv2.cvtColor(cv2.imread(sat_p), cv2.COLOR_BGR2RGB)
        mask = cv2.cvtColor(cv2.imread(mask_p), cv2.COLOR_BGR2RGB)
        
        sat = cv2.resize(sat, Config.IMG_SIZE, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, Config.IMG_SIZE, interpolation=cv2.INTER_NEAREST)
        
        # Extract features and labels
        features = extract_features(sat)
        labels = rgb_mask_to_label(mask, Config.COLOR_DICT)
        
        h, w, d = features.shape
        total_pixels = h * w
        sample_size = int(total_pixels * Config.SAMPLE_RATE)
        
        # Random sampling for computational efficiency
        rng = np.random.RandomState(Config.RANDOM_STATE + idx)
        sample_idx = rng.choice(total_pixels, sample_size, replace=False)
        
        X = features.reshape(-1, d)[sample_idx]
        y = labels.flatten()[sample_idx]
        
        return X, y
    except Exception as e:
        print(f"⚠ Error processing image {idx}: {e}")
        return None

print("Preparing dataset (this may take a few minutes)...")
t0 = time.time()

# Limit to MAX_IMAGES
n = min(len(sat_files), Config.MAX_IMAGES)
pairs = Parallel(n_jobs=-1, verbose=1)(delayed(process_one)(i, sat_files[i], mask_files[i]) for i in range(n))
pairs = [p for p in pairs if p is not None]

# Combine all samples
X_all = np.vstack([p[0] for p in pairs])
y_all = np.hstack([p[1] for p in pairs])

print(f"✓ Processed {n} images in {time.time()-t0:.1f}s")
print(f"✓ Total samples: {len(y_all):,}")
print(f"✓ Feature dimensions: {X_all.shape[1]}")

# Display class distribution
unique, counts = np.unique(y_all, return_counts=True)
print("\nClass Distribution:")
for cls, cnt in zip(unique, counts):
    if cls < len(Config.CLASS_NAMES):
        print(f"  {Config.CLASS_NAMES[cls]}: {cnt:,} ({100*cnt/len(y_all):.1f}%)")

In [ ]:
# ==========================================
# TRAIN-TEST SPLIT & SCALING
# ==========================================
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, 
    test_size=Config.TEST_SIZE, 
    random_state=Config.RANDOM_STATE,
    stratify=y_all
)

print(f"✓ Train: {len(y_train):,} samples")
print(f"✓ Test: {len(y_test):,} samples")

# Feature scaling using RobustScaler (less sensitive to outliers)
print("\nScaling features...")
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✓ Scaling complete")

In [ ]:
# ==========================================
# MODEL TRAINING: K-NEAREST NEIGHBORS
# ==========================================
print("Training K-Nearest Neighbors classifier...")
t0 = time.time()

knn = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
knn.fit(X_train_scaled, y_train)

print(f"✓ KNN trained in {time.time()-t0:.1f}s")

# Evaluate on test set
y_pred_knn = knn.predict(X_test_scaled)
knn_acc = accuracy_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn, average='weighted')

print(f"\nKNN Test Accuracy: {knn_acc:.4f}")
print(f"KNN Weighted F1-Score: {knn_f1:.4f}")

In [ ]:
# ==========================================
# MODEL TRAINING: RANDOM FOREST
# ==========================================
print("Training Random Forest classifier...")
t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    random_state=Config.RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)
rf.fit(X_train_scaled, y_train)

print(f"✓ Random Forest trained in {time.time()-t0:.1f}s")

# Evaluate on test set
y_pred_rf = rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf, average='weighted')

print(f"\nRandom Forest Test Accuracy: {rf_acc:.4f}")
print(f"Random Forest Weighted F1-Score: {rf_f1:.4f}")

In [ ]:
# ==========================================
# MODEL COMPARISON
# ==========================================
print("\n" + "="*50)
print("MODEL COMPARISON SUMMARY")
print("="*50)
print(f"KNN Accuracy:          {knn_acc:.4f}")
print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"\nImprovement: {(rf_acc - knn_acc)*100:.2f}%")
print("="*50)

# Detailed classification report for Random Forest
print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=Config.CLASS_NAMES, zero_division=0))

In [ ]:
# ==========================================
# CONFUSION MATRIX VISUALIZATION
# ==========================================
def plot_confusion_matrix(y_true, y_pred, title):
    """
    Plot a confusion matrix heatmap.
    
    Args:
        y_true (array): True labels
        y_pred (array): Predicted labels
        title (str): Plot title
    """
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=Config.CLASS_NAMES,
                yticklabels=Config.CLASS_NAMES)
    plt.title(title, fontsize=16, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()

# Plot confusion matrices for both models
plot_confusion_matrix(y_test, y_pred_knn, 'KNN Confusion Matrix')
plot_confusion_matrix(y_test, y_pred_rf, 'Random Forest Confusion Matrix')

In [ ]:
# ==========================================
# FEATURE IMPORTANCE ANALYSIS
# ==========================================
def plot_feature_importance(model, feature_names):
    """
    Visualize feature importance from Random Forest.
    
    Args:
        model: Trained Random Forest model
        feature_names (list): Names of features
    """
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(12, 6))
    plt.title('Random Forest Feature Importance', fontsize=16, fontweight='bold')
    plt.bar(range(len(importances)), importances[indices], color='steelblue')
    plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.xlabel('Features', fontsize=12)
    plt.ylabel('Importance', fontsize=12)
    plt.tight_layout()
    plt.show()

# Define feature names based on configuration
feature_names = ['Red', 'Green', 'Blue']
if Config.USE_SPECTRAL_INDICES:
    feature_names.extend(['NDVI', 'Brightness'])
if Config.USE_TEXTURE_FEATURES:
    feature_names.extend(['Texture_Std', 'Texture_Range'])

plot_feature_importance(rf, feature_names)

In [ ]:
# ==========================================
# VISUALIZATION: COMPREHENSIVE DASHBOARD
# ==========================================
import matplotlib.patches as mpatches

def visualize_comprehensive(idx):
    """
    Create a comprehensive visualization dashboard for a single image.
    Shows input, ground truth, predictions, and error analysis.
    
    Args:
        idx (int): Index of the image to visualize
    """
    print(f"Visualizing Image #{idx}: {os.path.basename(sat_files[idx])}...")
    
    # Load and resize data
    raw_sat = cv2.cvtColor(cv2.imread(sat_files[idx]), cv2.COLOR_BGR2RGB)
    raw_mask = cv2.cvtColor(cv2.imread(mask_files[idx]), cv2.COLOR_BGR2RGB)
    
    sat_img = cv2.resize(raw_sat, Config.IMG_SIZE, interpolation=cv2.INTER_LINEAR)
    mask_img = cv2.resize(raw_mask, Config.IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    
    true_labels = rgb_mask_to_label(mask_img, Config.COLOR_DICT)
    
    # Prepare features
    features = extract_features(sat_img)
    h, w, d = features.shape
    features_flat = features.reshape(-1, d)
    features_scaled = scaler.transform(features_flat)
    
    # Generate predictions
    knn_pred_flat = knn.predict(features_scaled)
    knn_pred_map = knn_pred_flat.reshape(h, w)
    
    rf_pred_flat = rf.predict(features_scaled)
    rf_pred_map = rf_pred_flat.reshape(h, w)
    
    # Colorize maps
    def colorize(labels):
        vis = np.zeros((h, w, 3))
        for i in range(7):
            if i < len(Config.CLASS_COLORS):
                vis[labels == i] = Config.CLASS_COLORS[i]
        return vis

    vis_true = colorize(true_labels)
    vis_knn = colorize(knn_pred_map)
    vis_rf  = colorize(rf_pred_map)
    
    # Create error map (White = Correct, Red = Wrong, Black = Unknown)
    error_map = np.ones((h, w, 3))
    errors = rf_pred_map != true_labels
    error_map[errors] = [1, 0, 0]
    ignore = true_labels == 6
    error_map[ignore] = [0, 0, 0]
    
    # Plot dashboard
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    axes[0,0].imshow(sat_img)
    axes[0,0].set_title("1. Satellite Image (Input)", fontsize=14, fontweight='bold')
    
    axes[0,1].imshow(vis_true)
    axes[0,1].set_title("2. Ground Truth (Goal)", fontsize=14, fontweight='bold')
    
    # Legend
    patches = [mpatches.Patch(color=Config.CLASS_COLORS[i], label=Config.CLASS_NAMES[i]) 
               for i in range(len(Config.CLASS_NAMES)-1)]
    axes[0,2].legend(handles=patches, loc='center', fontsize=12, title="Class Legend")
    axes[0,2].axis('off')
    axes[0,2].set_title("Legend", fontsize=14, fontweight='bold')

    axes[1,0].imshow(vis_knn)
    axes[1,0].set_title(f"3. KNN Prediction\n(Noisier)", fontsize=14)
    
    axes[1,1].imshow(vis_rf)
    axes[1,1].set_title(f"4. Random Forest Prediction\n(Smoother, more accurate)", fontsize=14, fontweight='bold')
    
    axes[1,2].imshow(error_map)
    axes[1,2].set_title("5. RF Error Map\n(Red = Incorrect)", fontsize=14, color='darkred')
    
    for ax in axes.flatten():
        if ax != axes[0,2]:
            ax.axis('off')
            
    plt.tight_layout()
    plt.show()

# Visualize examples
visualize_comprehensive(0) 
visualize_comprehensive(2)

In [ ]:
# ==========================================
# KNN CONFIDENCE ANALYSIS
# ==========================================
def visualize_knn_confidence(idx):
    """
    Analyze KNN prediction confidence using probability estimates.
    Darker areas indicate lower confidence (class boundaries).
    
    Args:
        idx (int): Index of the image to analyze
    """
    print(f"Analyzing KNN Confidence for Image #{idx}...")
    
    # Load and prepare data
    raw = cv2.cvtColor(cv2.imread(sat_files[idx]), cv2.COLOR_BGR2RGB)
    raw = cv2.resize(raw, Config.IMG_SIZE)
    truth_mask = cv2.resize(cv2.imread(mask_files[idx]), Config.IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    true_labels = rgb_mask_to_label(cv2.cvtColor(truth_mask, cv2.COLOR_BGR2RGB), Config.COLOR_DICT)
    
    # Extract features
    feats = extract_features(raw)
    h, w, d = feats.shape
    feats_scaled = scaler.transform(feats.reshape(-1, d))
    
    # Get probability estimates
    probs_flat = knn.predict_proba(feats_scaled)
    
    # Confidence is the maximum probability across classes
    confidence_map = np.max(probs_flat, axis=1).reshape(h, w)
    pred_map = np.argmax(probs_flat, axis=1).reshape(h, w)
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Prediction visualization
    vis_pred = np.zeros((h, w, 3))
    for c in range(7): 
        vis_pred[pred_map == c] = Config.CLASS_COLORS[c]
    axes[0].imshow(vis_pred)
    axes[0].set_title("KNN Prediction", fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Confidence heatmap
    im = axes[1].imshow(confidence_map, cmap='inferno', vmin=0.3, vmax=1.0)
    axes[1].set_title("KNN Confidence Map\n(Dark = Low Confidence)", fontsize=14, fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    # Ground truth
    vis_true = np.zeros((h, w, 3))
    for c in range(7): 
        vis_true[true_labels == c] = Config.CLASS_COLORS[c]
    axes[2].imshow(vis_true)
    axes[2].set_title("Ground Truth", fontsize=14, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Analyze confidence for a test image
visualize_knn_confidence(2)

In [ ]:
# ==========================================
# MODEL SAVING (OPTIONAL)
# ==========================================
# Uncomment to save trained models

# with open(os.path.join(Config.OUTPUT_PATH, 'rf_model.pkl'), 'wb') as f:
#     pickle.dump(rf, f)
# with open(os.path.join(Config.OUTPUT_PATH, 'knn_model.pkl'), 'wb') as f:
#     pickle.dump(knn, f)
# with open(os.path.join(Config.OUTPUT_PATH, 'scaler.pkl'), 'wb') as f:
#     pickle.dump(scaler, f)
# print("✓ Models saved successfully")